In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
from load_daie import load_daie

from adaptive_latents import ArrayWithTime, proSVD, KernelSmoother

In [ ]:
neural_data, stim = load_daie()

In [ ]:
pro = proSVD(k=2)
s = KernelSmoother(tau=0.2/neural_data.dt)
latents = pro.offline_run_on(s.offline_run_on(neural_data))
# latents = pro.offline_run_on(neural_data)



In [ ]:
from adaptive_latents.stim_regressor import StimRegressor, StimAutoReg, StreamingKalmanFilter

sr1 = StimRegressor(error_on_missed_stim=False, stim_delay=2*latents.dt, log_level=2, heed_stimuli=False, attempt_correction=False)

sr2 = StimRegressor(error_on_missed_stim=False, stim_delay=2*latents.dt, log_level=2, heed_stimuli=True, attempt_correction=True)
sr2.stim_autoreg = StimAutoReg(n_steps_to_consider=7)
sr2.stim_reg.length_scales = np.array([2.98538262e-01, 8.91250938e-01, 2.37137371e-07])
sr2.stim_reg.reweight_every = np.inf


sr1.offline_run_on([(latents,'X'),(stim,'stim')])

stim_shifted = deepcopy(stim)
stim_shifted.t = stim_shifted.t - 2 * latents.dt
sr2.offline_run_on([(latents,'X'), (stim_shifted, 'stim')])

srs = {
    'blind': sr1,
    'reg': sr2,
}

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(10,4))

for k, sr in srs.items():
    error = ArrayWithTime.from_list(sr.log['pred_error'], squeeze_type='to_2d')
    norm_error = ArrayWithTime(np.linalg.norm(error, axis=1),error.t)
    ax.plot(norm_error.t, norm_error, '-', label=f'{k} mse_whole={np.nanmean(norm_error ** 2):.2f} mse[800:]={np.nanmean(norm_error.slice_by_time(slice(800,None))** 2):.2f}')

ax.legend()

ax.set_xlabel('Time (s)')
ax.set_ylabel('norm error')

# for t in stim_shifted.t:
#     ax.axvline(t, color='red', alpha=0.3)

fig.savefig('/home/jgould/Downloads/daie21_1step.svg')



In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(8,8))

preq_errors = ArrayWithTime.from_list(srs['reg'].stim_reg.log['preq_errors'], squeeze_type='to_2d')
preq_errors = np.linalg.norm(preq_errors,axis=1)
n_observed = srs['reg'].stim_reg.n_observed


ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)
error_scatter = ax.scatter(srs['reg'].stim_reg.input_histories[0][:n_observed,0], srs['reg'].stim_reg.input_histories[0][:n_observed,1], c=preq_errors, cmap='plasma')
fig.colorbar(error_scatter)


In [ ]:

%matplotlib inline
fig, axs = plt.subplots(constrained_layout=True, figsize=(10,5), ncols=2)

n = 9
l = .1
r = 3
time_slice = slice(stim_shifted.t[n] -l, stim_shifted.t[n] + r)

ax = axs[1]
ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)
# ax.plot(latents[:,0], latents[:,1], '-')
idx = np.argmin(np.linalg.norm(latents - srs['reg'].stim_reg.input_histories[0][0],axis=1))
idx = latents.time_to_sample(srs['reg'].stim_reg.input_histories[2][:n_observed, 0]) -1
latents_at_stims = latents.slice(idx).slice_by_time(time_slice)
ax.scatter(latents_at_stims[:,0], latents_at_stims[:,1], c='r')
latent_slice = latents.slice_by_time(time_slice)
ax.plot(latent_slice[:,0], latent_slice[:,1])

ax = axs[0]
ax.plot(latent_slice.t, latent_slice, 'k');
for t in latents_at_stims.t:
    ax.axvline(t, color='r', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('latents')


In [ ]:
%matplotlib inline
from adaptive_latents.plotting_functions import AnimationManager

def plot_history_with_tail(ax, data, current_t, tail_length=1, scatter_all=True, dim_1=0, dim_2=1, hist_bins=None, invisible=False, scatter_alpha=.1, scatter_s=5):
    ax.cla()

    s = np.ones_like(data.t).astype(bool)
    if scatter_all:
        s = data.t <= current_t
    if hist_bins is None:
        ax.scatter(data[s,dim_1], data[s,dim_2], s=scatter_s, c='gray', edgecolors='none', alpha= 0 if invisible else scatter_alpha)
        forward_color = 'C1'
    else:
        s = s & np.isfinite(data).all(axis=1)
        ax.hist2d(data[s,dim_1], data[s,dim_2], bins=hist_bins)
        forward_color = 'white'


    linewidth = 4
    size = 10
    s = (current_t - tail_length < data.t) & (data.t <= current_t)
    # ax.plot(data[s, dim_1], data[s, dim_2], color=back_color, linewidth=linewidth * 1.5, alpha= 0 if invisible else 1)
    # ax.scatter(data[s, dim_1][-1], data[s, dim_2][-1], s=size * 1.5, color=back_color, alpha= 0 if invisible else 1)
    ax.plot(data[s, dim_1], data[s, dim_2], color=forward_color, linewidth=linewidth, alpha= 0 if invisible else 1)
    ax.scatter(data[s,dim_1][-1], data[s,dim_2][-1], color=forward_color, s=size, zorder=3, alpha= 0 if invisible else 1)
    ax.axis('off')


with AnimationManager('new_dataset_animation', outdir='.') as am:
    for t in np.linspace(1230, 1250, 20*20):
        ax = am.axs[0,0]
        ax.cla()

        plot_history_with_tail(ax, data=latents, current_t=t, tail_length=2, scatter_alpha=.9)



        idx = latents.time_to_sample(srs['reg'].stim_reg.input_histories[2][:n_observed, 0]) - 5
        idx = idx[(latents.time_to_sample(t - 2) <= idx) & (idx <= latents.time_to_sample(t))]
        ax.scatter(latents[idx,0], latents[idx,1], color='r', s=40, zorder=4)


        ax.set_title(f't={t:.2f}')

        am.grab_frame()

In [ ]:
idx = latents.time_to_sample(srs['reg'].stim_reg.input_histories[2][:n_observed, 0]) -1
idx
# (latents.time_to_sample(t - 2) <= idx) & (idx <= latents.time_to_sample(t))

In [ ]:
latents.time_to_sample(t-20) < idx